In [28]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import operator

In [29]:
load_dotenv()

True

In [30]:
model = ChatGroq(
    model="llama-3.1-8b-instant"
)

In [31]:
class ResumeEvaluation(BaseModel):
    score: int = Field(description="Score from 1 to 10")
    category: str
    strengths: list[str]
    weaknesses: list[str]
    recommendations: list[str]

In [32]:
structured_model = model.with_structured_output(ResumeEvaluation)

In [33]:
class ResumeState(TypedDict):
    resume: str
    evaluations: Annotated[list, operator.add]

In [34]:
def evaluate_ats(state: ResumeState):

    prompt = f"""
    Evaluate this resume from an ATS perspective.

    Resume:
    {state['resume']}

    Evaluate:
    - keyword usage
    - technical terminology
    - section structure
    - readability
    - ATS compatibility

    Give a score from 1 to 10.

    Identify strengths, weaknesses and specific recommendations.
    """

    result = structured_model.invoke(prompt)

    result.category = "ATS"

    return {
        "evaluations": [result]
    }

In [35]:
def evaluate_technical(state: ResumeState):

    prompt = f"""
    Evaluate the technical skills demonstrated in this resume.

    Resume:
    {state['resume']}

    Evaluate:
    - programming skills
    - frameworks
    - databases
    - AI/ML skills
    - cloud/devops
    - technical depth

    Give a score from 1 to 10.

    Identify strengths, weaknesses and specific recommendations.
    """

    result = structured_model.invoke(prompt)

    result.category = "Technical Skills"

    return {
        "evaluations": [result]
    }

In [36]:
def evaluate_projects(state: ResumeState):

    prompt = f"""
    Evaluate the projects mentioned in this resume.

    Resume:
    {state['resume']}

    Evaluate:
    - technical complexity
    - relevance
    - originality
    - measurable impact
    - clarity of project descriptions

    Give a score from 1 to 10.

    Identify strengths, weaknesses and specific recommendations.
    """

    result = structured_model.invoke(prompt)

    result.category = "Projects"

    return {
        "evaluations": [result]
    }

In [37]:
graph = StateGraph(ResumeState)

graph.add_node("ats", evaluate_ats)
graph.add_node("technical", evaluate_technical)
graph.add_node("projects", evaluate_projects)

graph.add_edge(START, "ats")
graph.add_edge(START, "technical")
graph.add_edge(START, "projects")

graph.add_edge("ats", END)
graph.add_edge("technical", END)
graph.add_edge("projects", END)

workflow = graph.compile()

In [38]:
initial_state = {
    "resume": """
    Ayush Vatsal

    Skills:
    Python, FastAPI, PostgreSQL, LangChain, LangGraph,
    RAG, React, Docker

    Projects:

    Senselytics:
    Built an AI-powered natural language SQL analytics platform
    using FastAPI, PostgreSQL and LangChain.

    RepoIntel:
    Built an AI codebase analysis system using LangGraph,
    FastAPI and vector search.

    Experience:
    Web Development Intern
    Worked on React and backend APIs.
    """
}

In [40]:
final_state = workflow.invoke(initial_state)
for evaluation in final_state["evaluations"]:
    print(evaluation)

score=8 category='ATS' strengths=['Good keyword usage', 'Technical terminology is relevant to the field', 'Section structure is clear'] weaknesses=['Readability can be improved', 'Some sections are not optimally structured for ATS'] recommendations=['Consider including a link to a portfolio or website', 'Use a more descriptive title for the experience section']
score=8 category='Projects' strengths=['Technical skills demonstrated', 'Measurable impact shown'] weaknesses=['Lack of depth in project descriptions', 'Limited context for some projects'] recommendations=['Improve clarity of project descriptions', 'Consider adding more technical details']
score=8 category='Technical Skills' strengths=['Strong understanding of programming concepts', 'Experience with multiple frameworks like FastAPI, React, and LangChain', 'Ability to work with databases like PostgreSQL', 'Strong AI/ML skills demonstrated through projects like Senselytics and RepoIntel'] weaknesses=['Limited experience with cloud